In [48]:

import pandas as pd
import argparse
import os
import sys
import mysql.connector
from dotenv import load_dotenv

load_dotenv()


True

In [49]:

DATA_DIR = "data"
SUB_IN = 1
SUB_OUT = 2
STARTING_LINEUP_PLAY_ID = 1
PLAYERS_PER_TEAM = 5
TECHNICAL = "Technical"


In [50]:

ID_COLS_PBP  = ["event_id", "play_id", "play_sequence", "period",
                "home_team_id", "away_team_id", "play_team_id"]
ID_COLS_PBPP = ["event_id", "play_id", "play_sequence", "period", "player_id", "team_id"]
ID_COLS_ROS  = ["event_id", "team_id", "player_id"]

def load(data_dir):
    pbp  = pd.read_excel(f"{data_dir}/pbp.xlsx")
    pbpp = pd.read_excel(f"{data_dir}/pbp-players.xlsx")
    ros  = pd.read_excel(f"{data_dir}/rosters.xlsx")

    pbpp = pbpp[pbpp.player_id.notna()].copy()

    pbp[ID_COLS_PBP]   = pbp[ID_COLS_PBP].astype(int)
    pbpp[ID_COLS_PBPP] = pbpp[ID_COLS_PBPP].astype(int)
    ros[ID_COLS_ROS]   = ros[ID_COLS_ROS].astype(int)

    pbp  = pbp.sort_values(["event_id", "play_id", "play_sequence"]).reset_index(drop=True)
    pbpp = pbpp.sort_values(["event_id", "play_id", "play_sequence"]).reset_index(drop=True)
    ros  = ros.sort_values(["event_id", "team_id", "player_id"]).reset_index(drop=True)
    return pbp, pbpp, ros



In [51]:

pbp, pbpp, ros = load(DATA_DIR)
pbp.shape, pbpp.shape, ros.shape


((1011, 30), (1284, 38), (66, 12))

In [52]:
def attach_roster_team(pbpp: pd.DataFrame, ros: pd.DataFrame) -> pd.DataFrame:
    roster_team = ros.set_index(["event_id", "player_id"])["team_id"].rename("roster_team_id")
    out = pbpp.join(roster_team, on=["event_id", "player_id"])
    out["team_id"] = out.pop("roster_team_id").astype(int)
    return out

In [53]:
pbpp = attach_roster_team(pbpp, ros)
print(pbpp.team_id.isna().sum(), "missing team_ids")
pbpp.loc[pbpp.play_id == 149, ["player_id", "last_name", "team_abbr", "team_id"]]

0 missing team_ids


,player_id,last_name,team_abbr,team_id
194,468895,Green,GS,9
195,173004,Iguodala,GS,9
835,845564,Booker,Hou,21


In [ ]:
OPENER_COLS = ["event_id", "period", "team_id", "player_id"]

def period_openers(pbpp: pd.DataFrame) -> pd.DataFrame:
    """Q1 from the Starting Lineup rows; Q2+ inferred from each player's first appearance."""

    starter = pbpp.loc[pbpp.play_id == STARTING_LINEUP_PLAY_ID, OPENER_COLS]

    evidence = pbpp[(pbpp.period >= 2) & (pbpp.play_detail != TECHNICAL)]
    first_row = evidence.drop_duplicates(subset=OPENER_COLS, keep="first")
    checked_in = (first_row.play_event == "Substitution") & (first_row.sequence == SUB_IN)
    inferred = first_row.loc[~checked_in, OPENER_COLS]

    openers = pd.concat([starter, inferred], ignore_index=True)
    return openers

In [77]:

openers = period_openers(pbpp)
openers.sort_values(["event_id", "period", "team_id", "player_id"])

,event_id,period,team_id,player_id
0,1947160,1,2,280587
9,1947160,1,2,469089
8,1947160,1,2,551768
7,1947160,1,2,883436
6,1947160,1,2,937647
...,...,...,...,...
70,1947312,4,21,467468
78,1947312,4,21,513526
71,1947312,4,21,567497
76,1947312,4,21,920066


In [78]:
counts = openers.groupby(["event_id", "period", "team_id"]).size().reset_index(name="players")
print("groups:", len(counts), "| all 5:", (counts.players == 5).all(), "| rows:", len(openers))
counts

groups: 16 | all 5: True | rows: 80


,event_id,period,team_id,players
0,1947160,1,2,5
1,1947160,1,9,5
2,1947160,2,2,5
3,1947160,2,9,5
4,1947160,3,2,5
5,1947160,3,9,5
6,1947160,4,2,5
7,1947160,4,9,5
8,1947312,1,10,5
9,1947312,1,21,5


In [ ]:
def openers_wide(openers: pd.DataFrame, ros: pd.DataFrame) -> pd.DataFrame:
    """One row per game per team"""

    named = openers.merge(ros[["event_id", "player_id", "name"]], on=["event_id", "player_id"])
    named["label"] = named.name + " (" + named.player_id.astype(str) + ")"

    per_period = (
        named.sort_values("name")
        .groupby(["event_id", "team_id", "period"])["label"]
        .agg(", ".join)
        .unstack("period")
        .rename(columns=lambda p: f"Q{p}")
        .reset_index()
    )
    return per_period

In [79]:
openers_stacked = openers_wide(openers,ros)
openers_stacked

period,event_id,team_id,Q1,Q2,Q3,Q4
0,1947160,2,"Al Horford (280587), Jaylen Brown (883436), Ja...","Aron Baynes (296379), Daniel Theis (1066725), ...","Al Horford (280587), Jaylen Brown (883436), Ja...","Aron Baynes (296379), Jayson Tatum (937647), M..."
1,1947160,9,"Draymond Green (468895), Kevin Durant (329525)...","Andre Iguodala (173004), David West (57934), D...","Draymond Green (468895), Kevin Durant (329525)...","Andre Iguodala (173004), David West (57934), D..."
2,1947312,10,"Chris Paul (229598), Clint Capela (830647), Ja...","Chris Paul (229598), Eric Gordon (401085), Nen...","Chris Paul (229598), Clint Capela (830647), Ja...","Chris Paul (229598), Eric Gordon (401085), Jam..."
3,1947312,21,"Devin Booker (845564), Greg Monroe (467468), M...","Alex Len (616982), Devin Booker (845564), Josh...","Devin Booker (845564), Greg Monroe (467468), M...","Dragan Bender (920066), Greg Monroe (467468), ..."


In [80]:
def sub_events(pbpp: pd.DataFrame) -> pd.DataFrame:
    """One row per substitution: event_id, play_id, team_id, player_in, player_out."""

    subs = pbpp[pbpp.play_event == "Substitution"]
    pivoted = (
        subs.pivot_table(index=["event_id", "play_id", "team_id"], columns="sequence", values="player_id", aggfunc="first")
        .rename(columns={SUB_IN: "player_in", SUB_OUT: "player_out"})
        .astype(int)
        .reset_index()
    )
    pivoted.columns.name = None
    return pivoted[["event_id", "play_id", "team_id", "player_in", "player_out"]]

In [81]:
subs = sub_events(pbpp)
subs

,event_id,play_id,team_id,player_in,player_out
0,1947160,41,2,697132,937647
1,1947160,59,9,173004,468895
2,1947160,66,2,792415,469089
3,1947160,67,9,263228,457611
4,1947160,96,2,1066725,551768
...,...,...,...,...,...
99,1947312,409,10,908263,329715
100,1947312,437,21,616982,467468
101,1947312,438,21,845574,567497
102,1947312,439,10,242307,395388


In [ ]:
OUT_COLS = ["event_id", "play_id", "player_id", "team_id", "period", "is_home"]

def walk_plays(pbp: pd.DataFrame, openers: pd.DataFrame, subs: pd.DataFrame) -> pd.DataFrame:
    """One row per play per on-court player"""

    plays = pbp[["event_id", "play_id", "period"]].drop_duplicates().sort_values(["event_id", "play_id"])

    rows = []
    for (eid, per, team), five in openers.groupby(["event_id", "period", "team_id"]):
        on_court = set(five.player_id)
        team_subs = subs[(subs.event_id == eid) & (subs.period == per) & (subs.team_id == team)].set_index("play_id")
        period_plays = plays[(plays.event_id == eid) & (plays.period == per)]

        for play in period_plays.itertuples(index=False):
            if play.play_id in team_subs.index:
                on_court.remove(team_subs.at[play.play_id, "player_out"])
                on_court.add(team_subs.at[play.play_id, "player_in"])

            rows.extend((eid, play.play_id, p, team, per) for p in on_court)

    long = pd.DataFrame(rows, columns=OUT_COLS[:-1])
    return add_is_home(long, pbp)

